# Ab Initio VQE Molecular Chemistry for H2

Variational Quantum Eigensolver simulation of the H2 molecule using PennyLane.

In [ ]:
import pennylane as qml
from pennylane import numpy as np

In [ ]:
symbols = ["H", "H"]

geometry = np.array([
    [0.0, 0.0, -0.6992],
    [0.0, 0.0,  0.6992]
], requires_grad=False)

hamiltonian, qubits = qml.qchem.molecular_hamiltonian(
    symbols,
    geometry,
    charge=0,
    mult=1,
    basis="sto-3g"
)

print("Qubits:", qubits)

In [ ]:
electrons = 2
hf_state = qml.qchem.hf_state(electrons, qubits)
singles, doubles = qml.qchem.excitations(electrons, qubits)

dev = qml.device("default.qubit", wires=qubits)

num_params = len(singles) + len(doubles)
params = np.zeros(num_params, requires_grad=True)

In [ ]:
@qml.qnode(dev)
def circuit(params):
    qml.BasisState(hf_state, wires=range(qubits))

    index = 0

    for wires in singles:
        qml.SingleExcitation(params[index], wires=wires)
        index += 1

    for wires in doubles:
        qml.DoubleExcitation(params[index], wires=wires)
        index += 1

    return qml.expval(hamiltonian)

In [ ]:
optimizer = qml.GradientDescentOptimizer(stepsize=0.2)

max_iterations = 100
tolerance = 1e-8

for iteration in range(max_iterations):
    params, previous_energy = optimizer.step_and_cost(circuit, params)
    energy = circuit(params)

    if iteration % 10 == 0:
        print(f"Iteration {iteration}: {energy:.8f} Hartree")

    if abs(energy - previous_energy) < tolerance:
        break

print("Final ground-state energy:", energy, "Hartree")